# geo0722 – Vector Analysis in Code: Climate Stations, Counties and Drinking Water Protection Zones

**Unit:** geo0720 Vector analysis · **Builds on:** geo0647/geo0646 (DWD station list), geo0221 (CRS in code), geo0441 (geometries, DVG1) · **QGIS twin:** geo0721 (the same questions with the Processing Toolbox) · **Data:** `data/geo0721/` (released with this notebook), the NRW counties (DVG1) from your course data folder or from opengeodata.nrw.de

Two questions about water and weather in North Rhine-Westphalia:

1. **How well do the DWD climate stations cover NRW?** How many active stations per county, and how much of each county is more than 20 km away from the nearest one?
2. **How much of NRW is drinking water protection zone?** Which counties have the largest share, and which climate stations stand inside a zone?

Every answer is one of four operations you know from the slides: **spatial join** (`sjoin`), **buffer**, **dissolve** and **overlay** (`overlay`: intersection, difference). In geo0721 you do the same in QGIS; compare the numbers.

## 0. Setup

In [ ]:
import os
import re
import warnings
from pathlib import Path

import pandas as pd
import geopandas as gpd
import shapely

def course_data(start=None):
    """The data/ folder of your course folder (outside the session folders, ignored by git).
    OPENGEO_DATA if set; else the folder above the session folder sNN_date; else ./data."""
    env = os.environ.get("OPENGEO_DATA")
    if env:
        return Path(env).expanduser()
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if re.fullmatch(r"s\d{2}_\d{4}-\d{2}-\d{2}", folder.name):
            return folder.parent / "data"
    return Path("data")


CRS = "EPSG:25832"                    # ETRS89 / UTM zone 32N: metres, the CRS of NRW geodata
DATA = course_data()                  # <course folder>/data: DVG1 lives here (geo0061)
HERE = Path("data/geo0721")           # small files released with this session
STATIONS_CSV = HERE / "dwd_kl_stations_nrw_active.csv"   # or your own file from geo0647/geo0646
WSG_GPKG = HERE / "wsg_nrw.gpkg"      # drinking water and spa protection zones NRW
RADIUS = 20_000                       # metres: "covered" = an active station within 20 km
OUT = Path("vector_analysis.gpkg")    # results for QGIS, next to this notebook

print("course data folder:", DATA.resolve())
print("geopandas", gpd.__version__, "| shapely", shapely.__version__)

## 1. Three layers

### 1.1 Stations: a CSV becomes a GeoDataFrame
The station list has longitude and latitude in degrees (WGS 84, EPSG:4326), as in geo0647. The station IDs stay **text** (`"05792"`, with the leading zero).

In [ ]:
stations = pd.read_csv(STATIONS_CSV, sep=";", dtype={"station_id": str}, parse_dates=["date_from", "date_to"])
stations = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations["lon"], stations["lat"]),
                            crs="EPSG:4326")           # assign: the numbers ARE degrees
print(len(stations), "active stations, CRS", stations.crs.to_string())
stations[["station_id", "name", "altitude", "geometry"]].head()

### 1.2 Counties from DVG1
The 53 counties of NRW: 31 *Kreise* and 22 independent cities (*kreisfreie Städte*), layer `dvg1krs_nw`. `read_dvg1` takes your local copy (geo0061) if it finds one, otherwise it reads the layer straight out of the zip file on opengeodata.nrw.de (about 20 s).

In [ ]:
DVG1_ZIP = "https://www.opengeodata.nrw.de/produkte/geobasis/vkg/dvg/dvg1/dvg1_EPSG25832_Shape.zip"
DVG1_LOCAL = [DATA / "dvg1_EPSG25832_Shape", Path.home() / "shared/data/opengeo/dvg1"]


def read_dvg1(layer="dvg1krs_nw", where=None):
    """A DVG1 layer (EPSG:25832) from a local copy or directly from the zip on opengeodata.nrw.de."""
    for folder in DVG1_LOCAL:
        if (folder / f"{layer}.shp").exists():
            print("local:", folder / f"{layer}.shp")
            return gpd.read_file(folder / f"{layer}.shp", where=where)
    print("reading from", DVG1_ZIP)
    return gpd.read_file(f"/vsizip//vsicurl/{DVG1_ZIP}", layer=layer, where=where)


counties = read_dvg1("dvg1krs_nw")[["KN", "GN", "ART", "geometry"]]   # KN: official key (text), GN: name
counties["area_km2"] = counties.area / 1e6
print(len(counties), "counties,", round(counties["area_km2"].sum()), "km², CRS", counties.crs.to_string())
counties.head(3)

### 1.3 Drinking water protection zones
Every protection area has zones: **1** the well field itself, **2** the 50-day line (bacteria die within 50 days of groundwater travel), **3** the whole catchment (often split into 3A, 3B, …). Spa protection areas (*Heilquellenschutzgebiete*, `prot_type` S) use letters A–D. `areastatus` says whether a zone is designated (F, *festgesetzt*) or still planned (P).

In [ ]:
wsg = gpd.read_file(WSG_GPKG)
print(len(wsg), "zones, CRS", wsg.crs.to_string())
print(pd.crosstab(wsg["zone_main"], [wsg["prot_type"], wsg["areastatus"]]))
wsg.head(3)

**Look first, then compute.** A quick map with all three layers. In the cell below, the stations must be **reprojected** (`to_crs`) to draw them on the same axes as the counties: plotting the degrees as metres would put all stations into one point near (0, 0).

In [ ]:
stations = stations.to_crs(CRS)             # transform: the coordinates change, the points stay where they are

ax = counties.plot(color="white", edgecolor="grey", linewidth=0.5, figsize=(9, 8))
wsg.plot(ax=ax, color="tab:blue", alpha=0.4)
stations.plot(ax=ax, color="red", markersize=25)
ax.set_title("Active DWD climate stations (red) and protection zones (blue), NRW")
ax.set_axis_off()

## 2. Spatial join: which county is each station in?
A **spatial join** adds the attributes of one layer to the other where the geometries match (here: point *within* polygon). The geometries are not changed. QGIS: *Vector general ▸ Join attributes by location*.

In [ ]:
joined = gpd.sjoin(stations, counties[["KN", "GN", "geometry"]], how="left", predicate="within")
print("stations without county:", joined["KN"].isna().sum())
joined[["station_id", "name", "GN", "KN"]].head()

Count per county and join the counts back to the county polygons (attribute join on the key `KN`). Counties without a station get no row in `counts`: fill them with 0.

In [ ]:
counts = joined.groupby("KN").size().rename("n_stations")
counties = counties.merge(counts, left_on="KN", right_index=True, how="left")
counties["n_stations"] = counties["n_stations"].fillna(0).astype(int)
counties["stations_per_1000km2"] = (counties["n_stations"] / counties["area_km2"] * 1000).round(2)
counties.sort_values("n_stations", ascending=False)[["GN", "ART", "n_stations", "area_km2", "stations_per_1000km2"]].head(8)

**Exercise 2.** How many counties have no active station? Which are the largest ones? Are they mostly rural counties (*Kreise*) or independent cities (*kreisfreie Städte*)? Look at the names and areas.

In [ ]:
# YOUR CODE HERE


## 3. Buffer, dissolve, difference: where is the next station more than 20 km away?

### 3.1 The pitfall: buffering in degrees
**Predict:** we take the stations in EPSG:4326 and buffer them by 0.2 (degrees). 0.2° of latitude is about 22 km. What shape do the buffers have on a map in metres, and how large are they east–west and north–south?

In [ ]:
in_degrees = stations.to_crs("EPSG:4326").buffer(0.2)        # geopandas warns, and rightly so
one = gpd.GeoSeries([in_degrees.iloc[0]], crs="EPSG:4326").to_crs(CRS).iloc[0]
xmin, ymin, xmax, ymax = one.bounds
print(f"east-west {(xmax - xmin) / 1000:.1f} km, north-south {(ymax - ymin) / 1000:.1f} km")
gpd.GeoSeries([one], crs=CRS).plot(edgecolor="black", facecolor="none").set_aspect("equal")

The "circle" is an **ellipse**: one degree of longitude is shorter than one degree of latitude (at 51.5° N: 69 km vs 111 km). Buffers, distances and areas are only meaningful in a **projected CRS with metres**. QGIS shows the unit next to the distance field of the *Buffer* tool: if it says *degrees*, stop and reproject.

### 3.2 The correct buffer, dissolved
In EPSG:25832 the distance is 20 000 m. The buffers of neighbouring stations overlap; **dissolve** (`union_all`) merges them into one area, so nothing is counted twice. QGIS: *Buffer* with *Dissolve result* ticked.

In [ ]:
covered = gpd.GeoDataFrame(geometry=[stations.buffer(RADIUS).union_all()], crs=CRS)
print(f"covered area: {covered.area.iloc[0] / 1e6:,.0f} km²")

### 3.3 Overlay: difference
`overlay(counties, covered, how="difference")` keeps the parts of each county that lie **outside** the covered area. Unlike the spatial join, an overlay **cuts** the geometries, and the result has new polygons. QGIS: *Vector overlay ▸ Difference*.

In [ ]:
gaps = gpd.overlay(counties[["KN", "GN", "geometry"]], covered, how="difference")
gaps["gap_km2"] = gaps.area / 1e6

counties = counties.merge(gaps[["KN", "gap_km2"]], on="KN", how="left")
counties["gap_km2"] = counties["gap_km2"].fillna(0)
counties["gap_pct"] = (100 * counties["gap_km2"] / counties["area_km2"]).round(1)

print(f"NRW: {100 * counties['gap_km2'].sum() / counties['area_km2'].sum():.1f} % more than {RADIUS / 1000:.0f} km from an active station")
counties.sort_values("gap_pct", ascending=False)[["GN", "n_stations", "gap_pct"]].head(8)

In [ ]:
ax = counties.plot(column="gap_pct", cmap="Reds", legend=True, edgecolor="grey", linewidth=0.4,
                    legend_kwds={"label": "% of the county > 20 km from an active station"}, figsize=(9, 8))
stations.plot(ax=ax, color="black", markersize=12)
ax.set_axis_off()

**Exercise 3.** Repeat with 10 km and 30 km (change `RADIUS` and rerun §3.2–3.3, or write a loop). How does the NRW share change? Why does a county with two stations still have gaps?

In [ ]:
# YOUR CODE HERE


## 4. Drinking water protection by county

### 4.1 Select and dissolve
We keep the **designated drinking water** zones (`areastatus` F, `prot_type` T). Zones of neighbouring protection areas can overlap: compare the sum of the zone areas with the area of the dissolved layer.

In [ ]:
drinking = wsg[(wsg["areastatus"] == "F") & (wsg["prot_type"] == "T")]
dissolved = gpd.GeoDataFrame(geometry=[drinking.union_all()], crs=CRS)
print(len(drinking), "zones")
print(f"sum of zone areas  {drinking.area.sum() / 1e6:,.0f} km²")
print(f"dissolved          {dissolved.area.iloc[0] / 1e6:,.0f} km²")
print(f"all zones (incl. planned and spa): {wsg.area.sum() / 1e6:,.0f} km² summed, {wsg.union_all().area / 1e6:,.0f} km² dissolved")

### 4.2 Overlay: intersection
`overlay(counties, dissolved, how="intersection")` cuts the protected area at the county boundaries: one piece per county. QGIS: *Vector overlay ▸ Intersection*, then the field calculator with `$area`.

In [ ]:
pieces = gpd.overlay(counties[["KN", "GN", "geometry"]], dissolved, how="intersection")
pieces["wsg_km2"] = pieces.area / 1e6

counties = counties.merge(pieces[["KN", "wsg_km2"]], on="KN", how="left")
counties["wsg_km2"] = counties["wsg_km2"].fillna(0)
counties["wsg_pct"] = (100 * counties["wsg_km2"] / counties["area_km2"]).round(1)

print(f"NRW: {100 * counties['wsg_km2'].sum() / counties['area_km2'].sum():.1f} % designated drinking water protection zone")
counties.sort_values("wsg_pct", ascending=False)[["GN", "ART", "wsg_km2", "wsg_pct"]].head(8)

(The warning about dropped geometries: where a zone only *touches* a county boundary, the intersection is a line or a point; overlay keeps polygons only, which is what we want here.)

### 4.3 Spatial join again: stations inside a zone
`how="inner"` keeps only the stations that fall into a zone. A station could appear twice if two zones overlap at its position.

In [ ]:
in_zone = gpd.sjoin(stations, drinking[["name", "wsg_zone", "zone_main", "geometry"]],
                    how="inner", predicate="within", lsuffix="station", rsuffix="wsg")
print(len(in_zone), "rows,", in_zone["station_id"].nunique(), "stations")
in_zone[["station_id", "name_station", "altitude", "name_wsg", "wsg_zone"]].sort_values("wsg_zone")

**Exercise 4.** Which counties have more than 20 % protection zone *and* no active climate station? Why could that matter for groundwater recharge estimates (which need precipitation)?

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

### 4.4 Nearest neighbour: how far is the next station from each protection area?
A **nearest join** (`sjoin_nearest`) attaches to every feature the closest feature of the other layer, and with `distance_col` the distance in CRS units (metres). First we dissolve the zones **by protection area** (`gebiet_id`): one polygon per area, the zones 1–3 merged. For a polygon, the distance is measured from its edge; a station inside the area has distance 0. QGIS: *Vector analysis ▸ Join attributes by nearest*.

In [ ]:
areas = drinking.dissolve(by=["gebiet_id", "name"], as_index=False)[["gebiet_id", "name", "geometry"]]
nearest = gpd.sjoin_nearest(areas, stations[["station_id", "name", "geometry"]], how="left",
                            distance_col="dist_m", lsuffix="wsg", rsuffix="station")
nearest["dist_km"] = (nearest["dist_m"] / 1000).round(1)
print(len(areas), "protection areas;", len(nearest), "rows (ties give more than one row)")
print(nearest["dist_km"].describe().round(1).to_string())
nearest.sort_values("dist_km", ascending=False)[["name_wsg", "name_station", "dist_km"]].head(5)

**Exercise 4.4.** How many protection areas are more than 20 km from the nearest active station? Which station is the nearest one for the most areas?

In [ ]:
# YOUR CODE HERE


## 5. Save for QGIS
One GeoPackage, three layers. Open it in QGIS next to your geo0721 results and compare the numbers.

In [ ]:
counties.to_file(OUT, layer="counties_stats")
stations.merge(joined[["station_id", "GN"]], on="station_id").to_file(OUT, layer="stations")
covered.to_file(OUT, layer="covered_20km")
print("written:", OUT.resolve(), gpd.list_layers(OUT)["name"].tolist())

## 6. Extension (optional): field parcels by the stream (Frohnenbruch, Kamp-Lintfort)
The same operations on a farm landscape: how much farmland lies in a strip along the streams? Two sources for "farmland": **OpenStreetMap** landuse and the official **field parcels** (*Teilschläge*) that farmers declare to the Chamber of Agriculture (Landwirtschaftskammer NRW) for EU payments, with the crop of the current year. The parcels come from a **WFS** of the state (the same idea as geo0921 Part C); if the server does not answer, the cell falls back to the copy in `data/geo0722/`.

Licences: OSM data © OpenStreetMap contributors (ODbL); field parcels and riparian strip search area: Landwirtschaftskammer NRW, dl-de/by-2-0.

In [ ]:
FROHN = Path("data/geo0722/frohnenbruch.gpkg")
LWK_WFS = "https://www.wfs.nrw.de/umwelt/lwk_eufoerderung"

aoi = gpd.read_file(FROHN, layer="aoi")
osm_landuse = gpd.read_file(FROHN, layer="osm_landuse")
streams = gpd.read_file(FROHN, layer="osm_water")
streams = streams[streams.geom_type.isin(["LineString", "MultiLineString"])]


def lwk_wfs(typename, bbox):
    """Features of the LWK WFS inside bbox (EPSG:25832)."""
    xmin, ymin, xmax, ymax = bbox
    url = (f"{LWK_WFS}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&SRSNAME=EPSG:25832"
           f"&TYPENAMES=umwelt_lwk_eufoerderung:{typename}"
           f"&BBOX={xmin:.0f},{ymin:.0f},{xmax:.0f},{ymax:.0f},urn:ogc:def:crs:EPSG::25832")
    return gpd.read_file(url)


try:
    parcels = lwk_wfs("Beantragte_und_als_foerderfaehig_festgestellte_Teilschlaege_in_NRW", aoi.total_bounds)
    parcels = parcels.clip(aoi)
    parcels = parcels[parcels.geom_type.isin(["Polygon", "MultiPolygon"])]
    print("WFS:", len(parcels), "parcels")
except Exception as err:                       # no network, server down, ...
    print("WFS failed:", type(err).__name__, "-> local copy")
    parcels = gpd.read_file(FROHN, layer="lwk_parcels")

print(f"AOI {aoi.area.iloc[0] / 1e4:.0f} ha | parcels {parcels.area.sum() / 1e4:.0f} ha | "
      f"OSM farmland+meadow {osm_landuse[osm_landuse['landuse'].isin(['farmland', 'meadow'])].area.sum() / 1e4:.0f} ha")
# AREA_HA is the area of the whole parcel; after clip() only the computed area is right
parcels.assign(ha=parcels.area / 1e4).groupby("USE_TXT").agg(n=("ha", "size"), ha=("ha", "sum")).round(1)

**Exercise 6.1.** Buffer the streams by 5 m (the riparian strip of the Federal Water Act, WHG §38a, on sloping arable land) and by 50 m, dissolve, and intersect with the parcels. Report hectares per use group (`USE_TXT`) for both widths. Then do the same with OSM farmland/meadow and compare.

In [ ]:
# YOUR CODE HERE


**Exercise 6.2.** The state publishes a *search area* for these strips (layer `Suchkulisse_WHG_DVO` of the same WFS): field blocks with a slope of at least 5 % next to water. How many are in our area, and why so few? Which streams does OSM know here, and which does it miss (look at the DOP in QGIS, geo0921 Part A)?

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

## Check yourself
* Spatial join or overlay: which one keeps the original geometries, which one cuts them? Which did you use for "stations per county", which for "protected area per county"?
* Why must the buffers be dissolved before you compute the covered area?
* You buffer by 0.01 in EPSG:4326. How large is the buffer, roughly, and why is that a problem?
* After a spatial join a station appears twice. What happened?